In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

print("Libraries imported successfully.")

In [ ]:
os.makedirs("plots", exist_ok=True)

print("Current Working Directory:")
print(os.getcwd())

In [ ]:
df = pd.read_csv("data/train.csv")

print("Dataset loaded successfully.")

In [ ]:
print("="*60)
print("DATASET OVERVIEW")
print("="*60)

print("\nShape:")
print(df.shape)

print("\nFirst Five Rows:")
display(df.head())

print("\nLast Five Rows:")
display(df.tail())

print("\nColumn Data Types:")
print(df.dtypes)

In [ ]:
print("="*60)
print("DATASET INFORMATION")
print("="*60)

df.info()

In [ ]:
missing_count = df.isnull().sum()

missing_percent = (missing_count / len(df)) * 100

missing_table = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percent.round(2)
})

missing_table = missing_table.sort_values(
    by="Missing Percentage",
    ascending=False
)

print(missing_table)

In [ ]:
high_missing = missing_table[
    missing_table["Missing Percentage"] > 20
]

print("Columns with more than 20% missing values:")

display(high_missing)

In [ ]:
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

rows_before = len(df)

df = df.drop_duplicates()

rows_after = len(df)

print("Rows Removed:", rows_before - rows_after)

print("New Shape:", df.shape)

In [ ]:
memory_before = df.memory_usage(deep=True).sum()

print("Memory Usage Before Conversion:")
print(f"{memory_before/1024:.2f} KB")

In [ ]:
df["MSSubClass"] = df["MSSubClass"].astype(str)

df["MSZoning"] = df["MSZoning"].astype("category")

In [ ]:
memory_after = df.memory_usage(deep=True).sum()

print("Memory Usage After Conversion:")
print(f"{memory_after/1024:.2f} KB")

print("\nMemory Saved:")
print(f"{(memory_before-memory_after)/1024:.2f} KB")

In [ ]:
# Columns where missing means feature is absent

no_feature_cols = [
    "Alley",
    "MasVnrType",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "MiscFeature"
]

for col in no_feature_cols:
    df[col] = df[col].fillna("NoFeature")

In [ ]:
# Numeric columns with less than 20% missing

numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:

    if df[col].isnull().sum() > 0:

        missing_percent = (df[col].isnull().sum()/len(df))*100

        if missing_percent < 20:

            df[col] = df[col].fillna(df[col].median())

In [ ]:
categorical_cols = df.select_dtypes(include=["object","category"]).columns

for col in categorical_cols:

    if df[col].isnull().sum() > 0:

        df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
print("="*50)

print("Remaining Missing Values")

print("="*50)

remaining = df.isnull().sum()

remaining = remaining[remaining>0]

print(remaining)

print("\nTotal Remaining Missing Values:")

print(df.isnull().sum().sum())

In [ ]:
display(df.describe())

In [ ]:
skewness = df.select_dtypes(include=np.number).skew()

skewness = skewness.sort_values(key=abs, ascending=False)

display(skewness)

In [ ]:
highest_skew = skewness.index[0]

print("Most Skewed Column:")

print(highest_skew)

print("Skewness:")

print(skewness.iloc[0])

In [ ]:
top2 = skewness.index[:2]

for col in top2:

    print("="*50)

    print(col)

    print("Mean :", df[col].mean())

    print("Median :", df[col].median())

In [ ]:
def iqr_outliers(column):

    Q1 = df[column].quantile(0.25)

    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR

    upper = Q3 + 1.5*IQR

    outliers = df[(df[column] < lower) | (df[column] > upper)]

    print(column)

    print("Q1:",Q1)

    print("Q3:",Q3)

    print("Outliers:",len(outliers))

    print("-"*50)

In [ ]:
iqr_outliers("SalePrice")

iqr_outliers("GrLivArea")

In [ ]:
##Section 3 — Visualizations
import os

os.makedirs("plots", exist_ok=True)

print("Plot folder ready")

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(df.index, df["SalePrice"])

plt.title("Sale Price Trend Across Dataset Rows")
plt.xlabel("Row Index")
plt.ylabel("Sale Price")

plt.savefig("plots/line_plot.png",
            bbox_inches="tight")

plt.show()

In [ ]:
neigh_mean = df.groupby("Neighborhood")["SalePrice"].mean()

plt.figure(figsize=(12,5))

plt.bar(neigh_mean.index,
        neigh_mean.values)

plt.xticks(rotation=90)

plt.title("Average Sale Price by Neighborhood")

plt.xlabel("Neighborhood")
plt.ylabel("Mean Sale Price")

plt.savefig("plots/bar_chart.png",
            bbox_inches="tight")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df["MiscVal"],
             bins=20)

plt.title("Distribution of MiscVal")

plt.xlabel("MiscVal")
plt.ylabel("Frequency")

plt.savefig("plots/histogram.png",
            bbox_inches="tight")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="GrLivArea",
    y="SalePrice"
)

plt.title("Relationship Between Living Area and Sale Price")

plt.xlabel("Above Ground Living Area")
plt.ylabel("Sale Price")

plt.savefig("plots/scatter_plot.png",
            bbox_inches="tight")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=df,
    x="OverallQual",
    y="SalePrice"
)

plt.title("Sale Price Distribution by Overall Quality")

plt.xlabel("Overall Quality")
plt.ylabel("Sale Price")

plt.savefig("plots/box_plot.png",
            bbox_inches="tight")

plt.show()

In [ ]:
# Select numeric columns
numeric_df = df.select_dtypes(include=np.number)

# Pearson correlation matrix
pearson_corr = numeric_df.corr()

# Select top correlated features with SalePrice
top_features = (
    pearson_corr["SalePrice"]
    .abs()
    .sort_values(ascending=False)
    .head(15)
    .index
)

# Create smaller correlation matrix
top_corr = pearson_corr.loc[top_features, top_features]


plt.figure(figsize=(12,10))

sns.heatmap(
    top_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    square=True
)

plt.title("Pearson Correlation Heatmap - Top 15 Features")

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()

plt.savefig(
    "plots/correlation_heatmap.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
pearson_corr = numeric_df.corr()

In [ ]:
# Remove self-correlation

corr_pairs = pearson_corr.abs().unstack()

corr_pairs = corr_pairs[corr_pairs < 1]

highest_pair = corr_pairs.idxmax()

highest_value = pearson_corr.loc[
    highest_pair[0],
    highest_pair[1]
]


print("Highest Correlation Pair:")
print(highest_pair)

print("\nCorrelation Value:")
print(highest_value)

In [ ]:
spearman_corr = numeric_df.corr(method="spearman")

display(spearman_corr)

In [ ]:
difference = abs(spearman_corr - pearson_corr)

# Remove diagonal

np.fill_diagonal(difference.values, 0)


diff_pairs = (
    difference
    .unstack()
    .sort_values(ascending=False)
)


top_three = diff_pairs.head(6)

print("Top Spearman-Pearson Differences:")

display(top_three)

In [ ]:
group_result = df.groupby(
    "Neighborhood"
)["SalePrice"].agg(
    ["mean","std","count"]
)


display(group_result)

In [ ]:
highest_mean_group = group_result["mean"].idxmax()

highest_std_group = group_result["std"].idxmax()


highest_mean = group_result["mean"].max()

lowest_mean = group_result["mean"].min()


ratio = highest_mean / lowest_mean


print("Highest Mean Group:")
print(highest_mean_group)


print("\nHighest Std Group:")
print(highest_std_group)


print("\nHighest Mean:")
print(highest_mean)


print("\nLowest Mean:")
print(lowest_mean)


print("\nRatio:")
print(ratio)

In [ ]:
df.to_csv(
    "cleaned_data.csv",
    index=False
)

print("cleaned_data.csv saved successfully")

In [ ]:
final_check = pd.read_csv("cleaned_data.csv")


print("Final Shape:")
print(final_check.shape)


print("\nFinal Missing Values:")
print(final_check.isnull().sum().sum())